In [ ]:
# Colab setup: mount Google Drive and configure project paths.
# Run this cell first on Colab. If running locally, skip this cell.
from google.colab import drive
drive.mount("/content/drive")

PROJECT_DIR = "/content/drive/MyDrive/Aini/ml-assignment/Team-Assignment-2/binus-ai-2026sem3-assignment2-group04"
MODEL_DIR   = f"{PROJECT_DIR}/models"
TUNER_DIR   = f"{PROJECT_DIR}/keras_tuner"
!mkdir -p "$MODEL_DIR"
!mkdir -p "$TUNER_DIR"

import tensorflow as tf
print("GPU:", tf.config.list_physical_devices("GPU"))
print("Project dir:", PROJECT_DIR)
print("Model dir:  ", MODEL_DIR)
print("Tuner dir:  ", TUNER_DIR)

# Group Assignment 2 — Phase 1C: Hyperparameter Tuning with Keras Tuner

**Objective:**
Search for an improved set of hyperparameters for the VGG-style baseline architecture from `01_baseline_EN.ipynb` (test acc 0.8735), using **Keras Tuner Hyperband**. The search space targets the knobs most likely to move the needle: learning rate, per-block dropout rates, L2 weight decay, dense head width, and optimizer choice.

**Search strategy — Hyperband (Li et al. 2017):**
Hyperband is an adaptive bandit-style allocator. Cheap candidates run for a few epochs; promising ones get more compute, weak ones get pruned early. With `max_epochs=20` and `factor=3` the algorithm explores ~30–40 distinct configurations while spending most of its budget on the top performers — far more efficient than grid or random search at the same compute cost.

**Why this comes after Phase 1B (transfer learning):**
We already know transfer learning (0.8993) outperforms the from-scratch baseline. The point of HP tuning is *not* to beat transfer learning — it can't, since both are bounded by 32×32 input scale on this architecture. The point is to:
1. **Establish how much of the baseline's headroom comes from HP choices vs. architecture.** If the tuned model gains ≥ 1 pp over hand-tuned, the manual HP was leaving easy wins on the table; if it's flat, our hand-tuning was already near-optimal for this architecture.
2. **Produce the third model for the report's comparison chapter** (`baseline_v1` → `baseline_tuned_v2` → `transfer_mobilenet_v1`), which is the centerpiece of the optimization narrative.

**Time budget:**
~1.5–2 hours on T4 GPU for the search, plus ~30 min to retrain the best HP for the full 50 epochs. Hyperband caches state on Drive (`keras_tuner/`), so an interrupted run can resume cleanly.

### Section 0: Setup and Reproducibility

Reuse `SEED = 42` from the baseline so the train/validation split is identical — the HP search and the baseline are both evaluated on bit-identical val sets, which is necessary for a fair comparison.

Hyperband itself introduces additional randomness (which trials get sampled, which get promoted) that isn't seeded. So if you re-run this notebook from scratch, the chosen HP will likely differ slightly. The cached `keras_tuner/` directory on Drive ensures resumability within a single search.

In [ ]:
SEED = 42
BATCH_SIZE = 64

import tensorflow as tf
from tensorflow.keras import layers, models, regularizers, callbacks
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json, pickle, os

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available:      {len(tf.config.list_physical_devices('GPU')) > 0}")
print(f"Random seed:        {SEED}")
print(f"Batch size:         {BATCH_SIZE}")

### Section 1: Load CIFAR-10 (Hugging Face)

Same Hugging Face source as the baseline notebook, for the same reason — the U Toronto mirror that Keras's built-in helper points at is unreliable. HF mirror is hosted on the HF Hub CDN.

In [ ]:
try:
    import datasets
except ImportError:
    !pip install -q datasets
    import datasets

print("Downloading CIFAR-10 via Hugging Face datasets...")
hf_dataset = datasets.load_dataset("cifar10")
hf_dataset.set_format("numpy")

train_images = np.array(hf_dataset['train']['img'])
train_labels = np.array(hf_dataset['train']['label']).reshape(-1, 1)
test_images  = np.array(hf_dataset['test']['img'])
test_labels  = np.array(hf_dataset['test']['label']).reshape(-1, 1)

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

print("Data loaded.")
print(f"Training samples: {len(train_images)}, test samples: {len(test_images)}")

### Section 2: Preprocessing and Stratified Train/Validation Split

Identical to the baseline pipeline (normalize → split with `SEED=42` → one-hot encode). Reusing the same val set ensures the HP-tuned model is evaluated on exactly the same examples as the baseline, so any accuracy delta is attributable to HP choices, not val-set luck.

In [ ]:
X_train_full = train_images.astype('float32') / 255.0
X_test       = test_images.astype('float32') / 255.0
y_train_full = train_labels.flatten()
y_test_int   = test_labels.flatten()

X_train, X_val, y_train_int, y_val_int = train_test_split(
    X_train_full, y_train_full,
    test_size=0.1, random_state=SEED, stratify=y_train_full,
)

y_train = to_categorical(y_train_int, num_classes=10)
y_val   = to_categorical(y_val_int,   num_classes=10)
y_test  = to_categorical(y_test_int,  num_classes=10)

print(f"X_train: {X_train.shape}, X_val: {X_val.shape}, X_test: {X_test.shape}")

### Section 3: Data Augmentation Pipeline

Same `tf.data` pipeline as the baseline: mild horizontal flip + small rotation + small zoom for training only; validation and test datasets pass through unchanged. Augmentation is fixed (not tuned) so that HP-attributable accuracy changes are not confounded with augmentation strength.

In [ ]:
augmenter = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
], name='augmenter')

train_ds = (tf.data.Dataset.from_tensor_slices((X_train, y_train))
            .shuffle(buffer_size=10000, seed=SEED)
            .batch(BATCH_SIZE)
            .map(lambda x, y: (augmenter(x, training=True), y),
                 num_parallel_calls=tf.data.AUTOTUNE)
            .prefetch(tf.data.AUTOTUNE))

val_ds = (tf.data.Dataset.from_tensor_slices((X_val, y_val))
          .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))

test_ds = (tf.data.Dataset.from_tensor_slices((X_test, y_test))
           .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))

print(f"train_ds: {len(X_train)} samples (augmented)")
print(f"val_ds:   {len(X_val)} samples")
print(f"test_ds:  {len(X_test)} samples")

### Section 4: HyperModel — Searchable Architecture

The architecture is the same VGG-style 3-block CNN from the baseline. We expose the following hyperparameters to Keras Tuner:

| Hyperparameter | Range | Sampling |
|---|---|---|
| `learning_rate` | 1e-4 – 1e-2 | log-uniform |
| `dropout_block1` | 0.1 – 0.4 | step 0.1 |
| `dropout_block2` | 0.2 – 0.5 | step 0.1 |
| `dropout_block3` | 0.3 – 0.5 | step 0.1 |
| `weight_decay` (L2) | 1e-5 – 1e-3 | log-uniform |
| `dense_units` | 64 / 128 / 256 | choice |
| `optimizer` | adam / adamw / sgd | choice |

That's 7 dimensions — large enough that grid search would be infeasible. Hyperband samples efficiently inside this product space.

The classifier-head dropout (the 0.5 before the final Dense) is **not** tuned — the head is the highest-overfitting risk point in the network and the literature standard 0.5 from Srivastava et al. (2014) is well-validated.

In [ ]:
try:
    import keras_tuner as kt
except ImportError:
    !pip install -q keras-tuner
    import keras_tuner as kt

print(f"Keras Tuner version: {kt.__version__}")


def build_model(hp):
    """HyperModel for Keras Tuner. Same VGG-style architecture as the
    baseline; only the listed knobs are exposed to the search."""
    weight_decay = hp.Float('weight_decay', 1e-5, 1e-3, sampling='log')
    dropout1 = hp.Float('dropout_block1', 0.1, 0.4, step=0.1)
    dropout2 = hp.Float('dropout_block2', 0.2, 0.5, step=0.1)
    dropout3 = hp.Float('dropout_block3', 0.3, 0.5, step=0.1)
    dense_units = hp.Choice('dense_units', [64, 128, 256])
    optimizer_name = hp.Choice('optimizer', ['adam', 'adamw', 'sgd'])
    learning_rate = hp.Float('learning_rate', 1e-4, 1e-2, sampling='log')

    model = models.Sequential([
        layers.Input(shape=(32, 32, 3)),

        # Block 1: 32 filters
        layers.Conv2D(32, (3, 3), padding='same', activation='relu',
                      kernel_regularizer=regularizers.l2(weight_decay)),
        layers.BatchNormalization(),
        layers.Conv2D(32, (3, 3), padding='same', activation='relu',
                      kernel_regularizer=regularizers.l2(weight_decay)),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(dropout1),

        # Block 2: 64 filters
        layers.Conv2D(64, (3, 3), padding='same', activation='relu',
                      kernel_regularizer=regularizers.l2(weight_decay)),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3, 3), padding='same', activation='relu',
                      kernel_regularizer=regularizers.l2(weight_decay)),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(dropout2),

        # Block 3: 128 filters
        layers.Conv2D(128, (3, 3), padding='same', activation='relu',
                      kernel_regularizer=regularizers.l2(weight_decay)),
        layers.BatchNormalization(),
        layers.Conv2D(128, (3, 3), padding='same', activation='relu',
                      kernel_regularizer=regularizers.l2(weight_decay)),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(dropout3),

        # Classifier head
        layers.Flatten(),
        layers.Dense(dense_units, activation='relu',
                     kernel_regularizer=regularizers.l2(weight_decay)),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(10, activation='softmax'),
    ])

    if optimizer_name == 'adam':
        opt = tf.keras.optimizers.Adam(learning_rate=learning_rate)
    elif optimizer_name == 'adamw':
        opt = tf.keras.optimizers.AdamW(learning_rate=learning_rate)
    else:  # sgd
        opt = tf.keras.optimizers.SGD(learning_rate=learning_rate, momentum=0.9)

    model.compile(optimizer=opt, loss='categorical_crossentropy', metrics=['accuracy'])
    return model

### Section 5: Hyperband Search

Hyperband settings:
- `max_epochs=20` — any single trial caps at 20 epochs (the search horizon).
- `factor=3` — successive halving ratio. Bracket sizes are `max_epochs / factor^k` for k = 0…s_max.
- `objective='val_accuracy'` — the metric that drives promotion decisions.

Total trials: roughly 30–40 (Hyperband decides automatically from the formula above). Total wall-clock time: ~1.5–2 hours on a T4. **The cell can be safely interrupted and re-run** — Keras Tuner persists state under `TUNER_DIR`, so the next call resumes from where it stopped.

Each trial uses an inner `EarlyStopping(patience=3)` to abort training early on already-bad candidates, saving ~30% of wall-clock at the long end of the bracket.

In [ ]:
tuner = kt.Hyperband(
    build_model,
    objective='val_accuracy',
    max_epochs=20,
    factor=3,
    directory=TUNER_DIR,
    project_name='cifar10_baseline_hpsearch',
    overwrite=False,    # resume cached results across re-runs
    seed=SEED,
)

tuner.search_space_summary()

print("\n" + "=" * 65)
print("Starting Hyperband search (this can take ~1.5-2 hours on T4 GPU)")
print("=" * 65)

tuner.search(
    train_ds,
    validation_data=val_ds,
    callbacks=[
        callbacks.EarlyStopping(monitor='val_loss', patience=3,
                                restore_best_weights=True, verbose=0),
    ],
    verbose=1,
)

print("\nSearch complete.")

### Section 6: Best Hyperparameters

We extract the top trial's hyperparameters and persist them to `models/best_hp.json` for the report and for future reproducibility. The next section retrains a fresh model with these HPs for the full 50-epoch budget (Hyperband only saw 20-epoch trials, so the final model can squeeze a bit more out of the same HP).

In [ ]:
best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]

print("Best hyperparameters found by Hyperband:")
print("-" * 65)
for k, v in best_hp.values.items():
    if isinstance(v, float):
        print(f"  {k:20s}: {v:.5g}")
    else:
        print(f"  {k:20s}: {v}")

# Persist to JSON for the report
hp_path = f"{MODEL_DIR}/best_hp.json"
with open(hp_path, "w") as f:
    json.dump(best_hp.values, f, indent=2)
print(f"\nSaved best HP to {hp_path}")

# Show the top 5 trials for context (helps the report discussion)
print("\nTop 5 trials by validation accuracy:")
print("-" * 65)
for i, trial in enumerate(tuner.oracle.get_best_trials(num_trials=5), 1):
    val_acc = trial.score
    hp = trial.hyperparameters.values
    print(f"  Trial {i}: val_acc = {val_acc:.4f}")
    print(f"    lr={hp.get('learning_rate'):.5g}, "
          f"opt={hp.get('optimizer')}, dense={hp.get('dense_units')}, "
          f"wd={hp.get('weight_decay'):.5g}")
    print(f"    drop={hp.get('dropout_block1'):.1f}/"
          f"{hp.get('dropout_block2'):.1f}/{hp.get('dropout_block3'):.1f}")

### Section 7: Retrain the Best Model for the Full 50 Epochs

Hyperband only saw each trial for at most 20 epochs. Now that we've identified the winning HP, we train a fresh model with those HPs on the full 50-epoch budget — same callbacks as the baseline (`EarlyStopping(patience=10)`, `ReduceLROnPlateau(factor=0.5, patience=3)`) so the training loop is bit-identical to `01_baseline_EN.ipynb` apart from the HP values themselves.

This separates "search compute" (cheap, exploratory) from "final compute" (focused on the best candidate), which is the standard Keras Tuner workflow.

In [ ]:
best_model = tuner.hypermodel.build(best_hp)
best_model.summary()

print("\nRetraining the best HP candidate for the full 50 epochs...")
print("-" * 65)

callback_list = [
    callbacks.EarlyStopping(monitor='val_loss', patience=10,
                            restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor='val_loss', patience=3,
                                factor=0.5, min_lr=1e-6, verbose=1),
]

history = best_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=callback_list,
    verbose=1,
)

print(f"\nFinal training finished after {len(history.history['loss'])} epoch(s).")
print(f"Best validation loss:     {min(history.history['val_loss']):.4f}")
print(f"Best validation accuracy: {max(history.history['val_accuracy']):.4f}")

### Section 8: Training History

Plot accuracy and loss curves the same way as the baseline notebook, so the two are visually comparable side-by-side in the report.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].set_title('Tuned Baseline — Model Accuracy')
axes[0].legend(loc='lower right'); axes[0].grid(alpha=0.3)

axes[1].plot(history.history['loss'], label='Train Loss', linewidth=2)
axes[1].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].set_title('Tuned Baseline — Model Loss')
axes[1].legend(loc='upper right'); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### Section 9: Test Set Evaluation

Same protocol as the baseline notebook: aggregate test loss / accuracy, then per-class precision, recall, and F1.

In [ ]:
print("Evaluating on the test set...")
test_loss, test_acc = best_model.evaluate(test_ds, verbose=0)
print(f"\nTest Loss:     {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}\n")

y_pred_proba = best_model.predict(test_ds, verbose=0)
y_pred = np.argmax(y_pred_proba, axis=1)
y_true = y_test_int

print("Per-class classification report:")
print("-" * 65)
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

### Section 10: Confusion Matrix

Visual overlap with the baseline confusion matrix tells us whether HP tuning shifted the *kind* of mistakes the model makes (e.g., reduced cat↔dog confusion specifically) or just made fewer mistakes uniformly. The latter is more typical for HP tuning that didn't change the architecture.

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names,
            cbar_kws={'label': 'Count'})
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix - Tuned Baseline (Hyperband HP) - CIFAR-10 Test Set')
plt.tight_layout()
plt.show()

### Section 11: High-Confidence Misclassifications

The 9 test images where the tuned model was most confidently wrong. Compared with the baseline's high-confidence misclassifications, the absolute confidence values often drop slightly after tuning — a side-effect of better-calibrated regularization.

In [ ]:
misclassified_mask = (y_pred != y_true)
misclassified_indices = np.where(misclassified_mask)[0]

predicted_confidence = y_pred_proba[misclassified_indices, y_pred[misclassified_indices]]
top9_local_idx = np.argsort(predicted_confidence)[-9:][::-1]
top9_global_idx = misclassified_indices[top9_local_idx]

plt.figure(figsize=(12, 12))
for i, idx in enumerate(top9_global_idx):
    plt.subplot(3, 3, i + 1)
    plt.imshow(X_test[idx])
    true_class = class_names[y_true[idx]]
    pred_class = class_names[y_pred[idx]]
    confidence = y_pred_proba[idx, y_pred[idx]]
    plt.title(f"{true_class} -> {pred_class}\n(confidence: {confidence:.1%})", fontsize=11)
    plt.axis('off')

plt.suptitle('Top 9 Most Confident Misclassifications (Tuned Baseline)', fontsize=14, y=1.00)
plt.tight_layout()
plt.show()

### Section 12: Three-Model Comparison Table

This is the canonical comparison the report's Chapter 6 will draw on. We hard-code the previously-known numbers for `baseline_v1` (0.8735) and `transfer_mobilenet_v1` (0.8993), and compute this notebook's `baseline_tuned_v2` numbers live.

In [ ]:
import pandas as pd

trainable_count = sum(int(tf.size(v).numpy()) for v in best_model.trainable_weights)

comparison = pd.DataFrame([
    {
        'Model': 'baseline_v1 (hand-tuned)',
        'Test Accuracy': 0.8735,
        'Trainable Params': '~570 K',
        'Notes': 'VGG-style, manual HP from 01_baseline_EN.ipynb',
    },
    {
        'Model': 'baseline_tuned_v2 (Hyperband)',
        'Test Accuracy': round(test_acc, 4),
        'Trainable Params': f'{trainable_count:,}',
        'Notes': 'Same architecture, Hyperband-searched HP',
    },
    {
        'Model': 'transfer_mobilenet_v1',
        'Test Accuracy': 0.8993,
        'Trainable Params': '~1.52 M',
        'Notes': 'MobileNetV2 ImageNet pretrained, fine-tuned at 96x96',
    },
])

print("Three-model comparison (this is the canonical table for report ch.6):")
print("=" * 75)
print(comparison.to_string(index=False))
print("=" * 75)
print(f"\nGap (tuned vs hand-tuned baseline):   {round((test_acc - 0.8735) * 100, 2):+} pp")
print(f"Gap (tuned vs transfer learning):     {round((test_acc - 0.8993) * 100, 2):+} pp")

### Section 13: Save Trained Model and History

Persist the tuned model and combined training history to Drive. The `best_hp.json` was already saved earlier in Section 6.

- `baseline_tuned_v2.keras` — full Keras v3 archive
- `baseline_tuned_v2_history.pkl` — `history.history` dict (for plotting in the report)
- `best_hp.json` — already saved in §6 (search-time artifact)

In [ ]:
model_path = f"{MODEL_DIR}/baseline_tuned_v2.keras"
best_model.save(model_path)
print(f"Model saved:   {model_path}  ({os.path.getsize(model_path) / 1e6:.1f} MB)")

history_path = f"{MODEL_DIR}/baseline_tuned_v2_history.pkl"
with open(history_path, "wb") as f:
    pickle.dump(history.history, f)
print(f"History saved: {history_path}")

print(f"\nAll Phase 1C artifacts:")
for f in ['baseline_tuned_v2.keras', 'baseline_tuned_v2_history.pkl', 'best_hp.json']:
    p = f"{MODEL_DIR}/{f}"
    if os.path.exists(p):
        print(f"  ✓ {p}  ({os.path.getsize(p) / 1e6:.2f} MB)")
    else:
        print(f"  ✗ {p}  (missing)")